<a href="https://colab.research.google.com/github/Rei5ende/pytorch-bc-study/blob/main/03_expert_data/cartpole_expert_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# -----------------------------------------------
# Random Play
# -----------------------------------------------

In [1]:
# -----------------------------------------------
# 1. 라이브러리 설치 및 임포트
# -----------------------------------------------

# gymnasium: OpenAI가 만든 강화학습 환경 라이브러리
# CartPole, MountainCar 등 다양한 환경 제공
!pip install gymnasium -q  # -q: 설치 로그 숨기기

import gymnasium as gym
import numpy as np
import torch

# -----------------------------------------------
# 2. CartPole 환경 생성
# -----------------------------------------------

# "CartPole-v1": 막대기가 달린 카트를 좌우로 움직여
# 막대기가 쓰러지지 않게 균형을 잡는 환경
env = gym.make("CartPole-v1")

# -----------------------------------------------
# 3. 환경 정보 확인
# -----------------------------------------------

# observation_space: 환경에서 받을 수 있는 관찰값의 형태
# CartPole의 관찰값 4개:
#   [0] 카트 위치       (-4.8 ~ 4.8)
#   [1] 카트 속도       (-무한 ~ 무한)
#   [2] 막대 각도       (-0.418 ~ 0.418 라디안, 약 ±24도)
#   [3] 막대 각속도     (-무한 ~ 무한)
print(f"관찰값 크기: {env.observation_space.shape}")  # (4,)

# action_space: 취할 수 있는 행동의 수
# CartPole의 행동 2개:
#   0: 카트를 왼쪽으로 밀기
#   1: 카트를 오른쪽으로 밀기
print(f"행동 개수: {env.action_space.n}")  # 2

# -----------------------------------------------
# 4. 랜덤 에이전트로 한 번 플레이
# -----------------------------------------------

# env.reset(): 환경 초기화, 첫 번째 관찰값 반환
# obs: 초기 상태의 관찰값 (4개 숫자)
# info: 추가 정보 (CartPole에서는 비어있음)
obs, info = env.reset()
total_reward = 0

for step in range(200):  # 최대 200 스텝 시도
    # env.action_space.sample(): 가능한 행동 중 랜덤으로 선택
    # → 0 또는 1 중 하나를 무작위로 반환
    action = env.action_space.sample()

    # env.step(action): 선택한 행동을 환경에 적용
    # 반환값:
    #   obs:       행동 후 새로운 관찰값
    #   reward:    이 스텝에서 받은 보상 (CartPole은 매 스텝 +1)
    #   done:      에피소드 종료 여부 (막대 쓰러짐 또는 카트 이탈)
    #   truncated: 최대 스텝 도달 여부 (200스텝 초과 시 True)
    #   info:      추가 정보
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward

    # done 또는 truncated가 True이면 에피소드 종료
    if done or truncated:
        print(f"랜덤 플레이 종료: {step+1} 스텝 버팀")
        break

print(f"총 보상: {total_reward}")

# env.close(): 환경 종료 및 메모리 해제
env.close()

관찰값 크기: (4,)
행동 개수: 2
랜덤 플레이 종료: 13 스텝 버팀
총 보상: 13.0


#시각화+저장

In [10]:
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from google.colab import files

env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, info = env.reset()
frames_random = []

for step in range(200):
    frame = env.render()
    frames_random.append(frame)

    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)

    if done or truncated:
        print(f"랜덤 플레이 종료: {step+1} 스텝 버팀")

        # 넘어진 후 20프레임 더 캡처
        for _ in range(20):
            frame = env.render()
            frames_random.append(frame)
        break

env.close()

# 애니메이션
fig, ax = plt.subplots(figsize=(6, 4))
ax.set_title("Random Agent")
ax.axis('off')
img = ax.imshow(frames_random[0])

def update_random(frame):
    img.set_data(frames_random[frame])
    return [img]

ani_random = animation.FuncAnimation(fig, update_random,
                                      frames=len(frames_random),
                                      interval=50, blit=True)
plt.close()

# GIF 저장 + 다운로드
#ani_random.save('/content/cartpole_random.gif', writer='pillow', fps=20)
#print("저장 완료!")
#files.download('/content/cartpole_random.gif')

HTML(ani_random.to_jshtml())

랜덤 플레이 종료: 23 스텝 버팀


# -----------------------------------------------
# 전문가 정책 (Expert Policy)
# -----------------------------------------------

In [8]:


def expert_policy(obs):
    """
    규칙 기반 전문가 에이전트

    핵심 아이디어:
    막대가 오른쪽으로 기울면 → 오른쪽으로 이동 (1)
    막대가 왼쪽으로 기울면  → 왼쪽으로 이동  (0)

    obs[2]: 막대 각도
    obs[3]: 막대 각속도 (기울어지는 속도)

    각도와 각속도를 함께 고려해야 더 안정적
    (각도만 보면 이미 늦을 수 있음)
    """
    pole_angle = obs[2]      # 막대 각도
    pole_velocity = obs[3]   # 막대 각속도

    # 각도와 각속도의 합이 양수면 오른쪽으로 기울고 있다는 뜻
    if pole_angle + 0.1 * pole_velocity > 0:
        return 1  # 오른쪽으로 이동
    else:
        return 0  # 왼쪽으로 이동

# -----------------------------------------------
# 전문가 에이전트 테스트
# -----------------------------------------------

env = gym.make("CartPole-v1")
obs, info = env.reset()
total_reward = 0

for step in range(200):
    action = expert_policy(obs)  # 전문가 정책으로 행동 선택
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward

    if done or truncated:
        print(f"전문가 플레이 종료: {step+1} 스텝 버팀")
        break

print(f"총 보상: {total_reward}")
env.close()

총 보상: 200.0


#시각화+저장

In [9]:
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from google.colab import files

# 전문가 정책
def expert_policy(obs):
    pole_angle = obs[2]
    pole_velocity = obs[3]
    if pole_angle + 0.1 * pole_velocity > 0:
        return 1
    else:
        return 0

env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, info = env.reset()
frames_expert = []

for step in range(200):
    frame = env.render()
    frames_expert.append(frame)

    action = expert_policy(obs)
    obs, reward, done, truncated, info = env.step(action)

    if done or truncated:
        print(f"전문가 플레이 종료: {step+1} 스텝 버팀")

        # 종료 후 10프레임 더 캡처
        for _ in range(10):
            frame = env.render()
            frames_expert.append(frame)
        break

env.close()

# 애니메이션
fig, ax = plt.subplots(figsize=(6, 4))
ax.set_title("Expert Agent")
ax.axis('off')
img = ax.imshow(frames_expert[0])

def update_expert(frame):
    img.set_data(frames_expert[frame])
    return [img]

ani_expert = animation.FuncAnimation(fig, update_expert,
                                      frames=len(frames_expert),
                                      interval=50, blit=True)
plt.close()

# GIF 저장 + 다운로드
#ani_expert.save('/content/cartpole_expert.gif', writer='pillow', fps=20)
#print("저장 완료!")
#files.download('/content/cartpole_expert.gif')

HTML(ani_expert.to_jshtml())

저장 완료!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



# -----------------------------------------------
# 전문가 데이터 수집 (states, actions 저장)
# -----------------------------------------------

In [11]:
# -----------------------------------------------
# 전문가 데이터 수집
# -----------------------------------------------

import numpy as np

env = gym.make("CartPole-v1")

# 데이터를 저장할 리스트
states = []   # 관찰값 (각 스텝의 환경 상태)
actions = []  # 행동 (전문가가 선택한 행동)

# 여러 에피소드 반복해서 데이터 수집
num_episodes = 50  # 50번 플레이
total_steps = 0

for episode in range(num_episodes):
    obs, info = env.reset()

    for step in range(200):
        # 현재 상태 저장
        states.append(obs)

        # 전문가 행동 결정
        action = expert_policy(obs)

        # 행동 저장
        actions.append(action)

        obs, reward, done, truncated, info = env.step(action)
        total_steps += 1

        if done or truncated:
            break

env.close()

# 리스트 → numpy 배열로 변환
states = np.array(states)    # shape: (전체 스텝 수, 4)
actions = np.array(actions)  # shape: (전체 스텝 수,)

print(f"수집된 에피소드 수: {num_episodes}")
print(f"수집된 전체 스텝 수: {total_steps}")
print(f"states shape: {states.shape}")
print(f"actions shape: {actions.shape}")
print(f"\n첫 번째 state: {states[0]}")
print(f"첫 번째 action: {actions[0]}")

수집된 에피소드 수: 50
수집된 전체 스텝 수: 10000
states shape: (10000, 4)
actions shape: (10000,)

첫 번째 state: [ 0.00679997 -0.00795248 -0.04157381 -0.03535334]
첫 번째 action: 0


# 전문가 데이터 수집 결과

## 수집 설정

*   **에피소드 수**: 50회
*   **에피소드당 최대 스텝**: 200

## 수집 결과

*   **전체 스텝 수**: 10,000
*   전문가는 매 에피소드마다 200 스텝을 모두 버텼습니다.
*   랜덤 에이전트(10~44 스텝)와 비교하면 압도적인 차이를 보입니다.

## 데이터 구조

*   `states` (10000, 4) — 각 스텝의 환경 상태 (카트 위치, 카트 속도, 막대 각도, 막대 각속도)
*   `actions` (10000,) — 각 스텝에서 전문가가 선택한 행동 (0: 왼쪽, 1: 오른쪽)

### 첫 번째 데이터 예시

*   **state**: `[ 0.00679997 -0.00795248 -0.04157381 -0.03535334]`
*   **action**: `0` → 왼쪽으로 이동
*   막대가 왼쪽으로 약간 기울어져 있어서 왼쪽으로 이동해 균형을 잡으려는 것을 알 수 있습니다.

## 다음 단계

이 10,000개의 (state, action) 쌍이 Behavior Cloning 학습의 재료가 됩니다.